In [13]:
# Cell 1: Imports
import time
import hashlib
from random import randint

In [14]:
# Cell 2: Parameter Generation

def generate_params(bits=32):
    """
    Generates parameters for Schnorr:
    A prime p and a generator g of the group.
    """
    # 1. Generate a 'Safe Prime' p where p = 2q + 1 and q is also prime.
    # This prevents the Pohlig-Hellman algorithm from attacking the DLP.
    
    # We use Sage's 'is_prime' to ensure the prime is strong.
    found = False
    while not found:
        # Generate a random prime q
        q = random_prime(2^bits, lbound=2^(bits-1))
        p = 2*q + 1
        if is_prime(p):
            found = True
            
    # 2. Find a generator g
    # In a safe prime group, we often want a generator of the 
    # subgroup of quadratic residues to ensure maximum security.
    # Sage's primitive_root works perfectly here.
    g = primitive_root(p)
    
    return p, g

In [15]:
# Cell 3: Key Generation

def keygen(p, g):
    x = randint(1, p-2)          # private key
    y = power_mod(g, x, p)       # public key
    return x, y

In [16]:
# Cell 4: Hash Function

def hash_function(r, m, p):
    # Encoding to bytes ensures consistency across different systems
    data = f"{r}|{m}".encode()
    h = hashlib.sha256(data).hexdigest()
    # We take the hash modulo (p-1) to ensure the exponent stays in range
    return int(h, 16) % (p - 1)

In [17]:
# Cell 5: Signing

def sign(p, g, x, m):
    # k must be coprime to p-1 for maximum security, but random is okay for demo
    k = randint(1, p - 2)
    
    # Commitment
    r = power_mod(g, k, p)
    
    # Challenge
    e = hash_function(r, m, p)
    
    # Response: s = (k + x*e) mod (p-1)
    s = (k + x * e) % (p - 1)
    
    print(r)

In [18]:
# Cell 6: Verification

def verify(p, g, y, m, r, s):
    e = hash_function(r, m, p)
    
    # Verification equation: g^s == r * y^e (mod p)
    left_side = power_mod(g, s, p)
    right_side = (r * power_mod(y, e, p)) % p
    
    return left_side == right_side

In [19]:
# Cell 7: Main Execution (Detailed + Timing)

import time

print("🔐 Schnorr Signature Scheme Demo\n")

# Step 1: Parameters
p, g = generate_params()

# Recover q (since p = 2q + 1)
q = (p - 1) // 2

# Step 2: Keys
x, y = keygen(p, g)

print("📌 Parameters:")
print("p (prime):", p)
print("q (subgroup prime):", q)
print("g (generator):", g)

print("\n🔑 Keys:")
print("Private Key (x):", x)
print("Public Key (y):", y)

print("--------------------------------------------------")

# Step 3: User Input
m = input("✍️ Enter your message: ")

start_time = time.time()
# Step 4: Signing
k = randint(1, p - 2)  # expose k for learning

r = power_mod(g, k, p)
e = hash_function(r, m, p)
s = (k + x * e) % (p - 1)

print("\n🧮 Signing Variables:")
print("Random k:", k)
print("r (commitment):", r)
print("e (hash):", e)
print("s (response):", s)

print("--------------------------------------------------")

# Step 5: Verification
left_side = power_mod(g, s, p)
right_side = (r * power_mod(y, e, p)) % p

valid = (left_side == right_side)

print("\n🔍 Verification:")
print("g^s mod p:", left_side)
print("r * y^e mod p:", right_side)

print("--------------------------------------------------")

# Time End
end_time = time.time()

print("\n⏱️ Execution Time:", round(end_time - start_time, 6), "seconds")

# Final Result
if valid:
    print("✅ Signature is VALID")
else:
    print("❌ Signature is INVALID")

🔐 Schnorr Signature Scheme Demo

📌 Parameters:
p (prime): 6995127467
q (subgroup prime): 3497563733
g (generator): 2

🔑 Keys:
Private Key (x): 2336632058
Public Key (y): 6559102283
--------------------------------------------------


✍️ Enter your message:  hi



🧮 Signing Variables:
Random k: 4938065006
r (commitment): 3011097854
e (hash): 4805220444
s (response): 5070157610
--------------------------------------------------

🔍 Verification:
g^s mod p: 3030145934
r * y^e mod p: 3030145934
--------------------------------------------------

⏱️ Execution Time: 0.000726 seconds
✅ Signature is VALID


In [12]:
def on_button_clicked(b):
    with output:
        clear_output()
        import time
        start_time = time.time()
        
        # 1. Params
        p_val, g_val = generate_params(bits=bit_slider.value)
        q_val = (p_val - 1) // 2
        
        # 2. Keys
        x_val, y_val = keygen(p_val, g_val)
        
        # 3. Signing
        k_val = randint(1, p_val - 2)
        r_val = power_mod(g_val, k_val, p_val)
        e_val = hash_function(r_val, msg_input.value, p_val)
        s_val = (k_val + x_val * e_val) % (p_val - 1)
        
        # 4. Verify
        left = power_mod(g_val, s_val, p_val)
        right = (r_val * power_mod(y_val, e_val, p_val)) % p_val
        is_valid = (left == right)
        
        end_time = time.time()

        print("📌 Parameters:")
        print(f"p: {p_val}")
        print(f"q: {q_val}")
        print(f"g: {g_val}")
        
        print("\n🔑 Keys:")
        print(f"x (private): {x_val}")
        print(f"y (public): {y_val}")
        
        print("\n🧮 Signing:")
        print(f"k: {k_val}")
        print(f"r: {r_val}")
        print(f"e: {e_val}")
        print(f"s: {s_val}")
        
        print("\n🔍 Verification:")
        print(f"g^s mod p: {left}")
        print(f"r * y^e mod p: {right}")
        
        print(f"\n⏱️ Time: {round(end_time - start_time, 6)} sec")
        
        print(f"\nResult: {'✅ Valid' if is_valid else '❌ Invalid'}")